In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
train_real = tf.data.Dataset.list_files("/content/drive/MyDrive/face_detection_train/face_real/*.jpg",shuffle=True)
train_fake = tf.data.Dataset.list_files("/content/drive/MyDrive/face_detection_train/face_fake/*.jpg",shuffle=True)
test_real = tf.data.Dataset.list_files( "/content/drive/MyDrive/face_detection_test/face_real/*.jpg",shuffle=False)
test_fake = tf.data.Dataset.list_files("/content/drive/MyDrive/face_detection_test/face_fake/*.jpg",shuffle=False)


In [ ]:
def load_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    return img
train_real = train_real.map(load_image)
train_fake = train_fake.map(load_image)
test_real = test_real.map(load_image)
test_fake = test_fake.map(load_image)

In [ ]:
def add_label(image, label):
  return image , label

train_real = train_real.map(lambda x: add_label(x,0))
train_fake = train_fake.map(lambda x: add_label(x,1))
test_real = test_real.map(lambda x: add_label(x, 0))
test_fake = test_fake.map(lambda x: add_label(x, 1))

In [ ]:
train_dataset = train_real.concatenate(train_fake)
test_dataset = test_real.concatenate(test_fake)
train_dataset = train_dataset.shuffle(8000)
#test_dataset = test_dataset.shuffle(2000)
train_size = int(0.8 * 8000)   # 6400
val_size = 8000 - train_size   # 1600

validation_dataset = train_dataset.skip(train_size)
train_dataset = train_dataset.take(train_size)

In [ ]:
from tensorflow.keras.applications.efficientnet import preprocess_input
def preprocess(image, label):
    image = preprocess_input(image)
    return image, label
train_dataset = train_dataset.map(preprocess)
validation_dataset = validation_dataset.map(preprocess)
test_dataset = test_dataset.map(preprocess)


In [ ]:
BATCH_SIZE = 32
train_dataset = train_dataset.batch(BATCH_SIZE)
validation_dataset = validation_dataset.batch(BATCH_SIZE)
test_dataset = test_dataset.batch(BATCH_SIZE)
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(AUTOTUNE)
validation_dataset = validation_dataset.prefetch(AUTOTUNE)
test_dataset = test_dataset.prefetch(AUTOTUNE)

In [ ]:
model = tf.keras.models.load_model("/content/drive/MyDrive/efficientnet_stage1.keras")
base_model = model.layers[1]
base_model.trainable = True
for layers in base_model.layers[:-30]:
  layers.trainable = False


In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
              loss="binary_crossentropy",
              metrics=["accuracy"])


In [ ]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,050,852 (15.45 MB)

 Trainable params: 1,497,441 (5.71 MB)

 Non-trainable params: 2,553,411 (9.74 MB)

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "/content/drive/MyDrive/efficientnet_finetuned.keras",
    monitor="val_accuracy",
    save_best_only=True
)

history_finetune = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=15,
    callbacks=[early_stop, checkpoint]
)

Epoch 1/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 820s 254ms/step - accuracy: 0.5895 - loss: 0.6714 - val_accuracy: 0.6432 - val_loss: 0.6450
Epoch 2/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 54s 162ms/step - accuracy: 0.6177 - loss: 0.6537 - val_accuracy: 0.6928 - val_loss: 0.6204
Epoch 3/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 55s 163ms/step - accuracy: 0.6403 - loss: 0.6381 - val_accuracy: 0.7150 - val_loss: 0.6018
Epoch 4/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 55s 166ms/step - accuracy: 0.6692 - loss: 0.6172 - val_accuracy: 0.7378 - val_loss: 0.5810
Epoch 5/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 56s 169ms/step - accuracy: 0.6961 - loss: 0.5987 - val_accuracy: 0.7411 - val_loss: 0.5632
Epoch 6/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 58s 176ms/step - accuracy: 0.7095 - loss: 0.5831 - val_accuracy: 0.7418 - val_loss: 0.5567
Epoch 7/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 56s 163ms/step - accuracy: 0.7278 - loss: 0.5702 - val_accuracy: 0.7532 - val_loss: 0.5386
Epoch 8/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 55s 164ms/step - accuracy: 0.7344 - loss: 

In [ ]:
model.evaluate(test_dataset)

63/63 ━━━━━━━━━━━━━━━━━━━━ 6s 93ms/step - accuracy: 0.6585 - loss: 0.6373


[0.6373271942138672, 0.6585000157356262]

In [ ]:
test_real = tf.data.Dataset.list_files(
    "/content/drive/MyDrive/face_detection_test/face_real/*.jpg",
    shuffle=False
)

test_fake = tf.data.Dataset.list_files(
    "/content/drive/MyDrive/face_detection_test/face_fake/*.jpg",
    shuffle=False
)

test_real = test_real.map(load_image)
test_fake = test_fake.map(load_image)

test_real = test_real.map(lambda x: (x,0))
test_fake = test_fake.map(lambda x: (x,1))

test_dataset = test_real.concatenate(test_fake)

test_dataset = test_dataset.batch(32)
test_dataset = test_dataset.prefetch(tf.data.AUTOTUNE)

In [ ]:
test_loss, test_acc = model.evaluate(test_dataset)

print(test_loss, test_acc)

63/63 ━━━━━━━━━━━━━━━━━━━━ 10s 88ms/step - accuracy: 0.6585 - loss: 0.6373
0.6373271942138672 0.6585000157356262


In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

y_true = []
for _, labels in test_dataset:
    y_true.extend(labels.numpy())

y_true = np.array(y_true)

y_pred_prob = model.predict(test_dataset)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

print(confusion_matrix(y_true,y_pred))

print(classification_report(
    y_true,
    y_pred,
    digits=4
))

63/63 ━━━━━━━━━━━━━━━━━━━━ 5s 86ms/step
[[574 426]
 [257 743]]
              precision    recall  f1-score   support

           0     0.6907    0.5740    0.6270      1000
           1     0.6356    0.7430    0.6851      1000

    accuracy                         0.6585      2000
   macro avg     0.6632    0.6585    0.6560      2000
weighted avg     0.6632    0.6585    0.6560      2000



In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-60]:
    layer.trainable = False

In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-6),
              loss="binary_crossentropy",
              metrics=["accuracy"])


In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "/content/drive/MyDrive/efficientnet_finetuned2.keras",
    monitor="val_accuracy",
    save_best_only=True
)

history_finetune = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=10,
    callbacks=[early_stop, checkpoint]
)

Epoch 1/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 93s 220ms/step - accuracy: 0.7917 - loss: 0.4650 - val_accuracy: 0.8370 - val_loss: 0.3996
Epoch 2/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 51s 154ms/step - accuracy: 0.7873 - loss: 0.4656 - val_accuracy: 0.8236 - val_loss: 0.4152
Epoch 3/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 82s 153ms/step - accuracy: 0.7892 - loss: 0.4585 - val_accuracy: 0.8236 - val_loss: 0.4193
Epoch 4/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 52s 155ms/step - accuracy: 0.7973 - loss: 0.4595 - val_accuracy: 0.8068 - val_loss: 0.4264


In [ ]:
model = tf.keras.models.load_model(
    "/content/drive/MyDrive/efficientnet_finetuned2.keras"
)
test_loss, test_acc = model.evaluate(test_dataset)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_acc)

63/63 ━━━━━━━━━━━━━━━━━━━━ 20s 169ms/step - accuracy: 0.6575 - loss: 0.6374
Test Loss: 0.6374036073684692
Test Accuracy: 0.6575000286102295
